In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

#### State Backend

In [2]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [3]:
# 1. create the agent - these two are equivalent
agent = create_deep_agent(model="google_genai:gemini-2.5-flash-lite")

# under the hood this is what agent is doing - explicit state backend
agent2 = create_deep_agent(
    model="google_genai:gemini-2.5-flash-lite",
    backend=StateBackend()
)

In [4]:
# 2. invoke the agent and ask it to write a file (statebackend keeps that file inside langgraph state)
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you have done it"
        )
    }]
})

print("\n--- Agent reply ---")
print(result["messages"][-1].content)


--- Agent reply ---
I have created the file `/notes/todo.txt` with the content you provided.


In [5]:
result

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt with exactly this content:\n1. record video\n2. Edit video\n3. Upload video\nThen tell me you have done it', additional_kwargs={}, response_metadata={}, id='f3a4616b-ed4c-41ac-9bca-a8eb2eef6007'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"content": "1. record video\\n2. Edit video\\n3. Upload video", "file_path": "/notes/todo.txt"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a062b7-716d-7a71-b6de-ad84bc5fba38-0', tool_calls=[{'name': 'write_file', 'args': {'content': '1. record video\n2. Edit video\n3. Upload video', 'file_path': '/notes/todo.txt'}, 'id': 'f92b2c3b-e303-48f9-8286-da9a0ec72ee4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2329, 'output_tokens': 41, 'total_tokens': 2370, 'input_token_details': {'cache_

In [6]:
# 3. check backend is working with state backend written files appear under result["files"]

print("--- Backend check ---")
files = result.get("files", [])

if files:
    print(f"StateBackend is working - {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n{path} \n{'-'*10}\n{content}")
else:
    print("No files found in state")

--- Backend check ---
StateBackend is working - 1 file(s) in state:

/notes/todo.txt 
----------
{'content': '1. record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-09-02T15:23:07.387816+00:00', 'modified_at': '2026-09-02T15:23:07.387816+00:00'}


#### FileSystemBackend (local disk)

In [7]:
# 1. create the agent with real disk backend
# root_dir -> file land relative to your current working directory
# virtual_mode = True -> agent uses virtual paths like /notes/todo.txt mapped onto root dir

from deepagents.backends import FilesystemBackend
ROOT = "."

agent = create_deep_agent(model="google_genai:gemini-2.5-flash-lite", backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True))
print(f"Agent created with FilesSystemBackend (root_dir={ROOT})")
print("Files written by agent will appear on actual disk")

Agent created with FilesSystemBackend (root_dir=.)
Files written by agent will appear on actual disk


In [10]:
# 2. invoke the agent and ask it to write a file
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you have done it"
        )
    }]
})

print("\n--- Agent reply ---")
print(result["messages"][-1].content)


--- Agent reply ---
I have created the file /notes/todo.txt with the content you provided.


In [11]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 D:\StudyAndWork\GenAI-AgenticAI\learndeepagents\src\notes\todo.txt
----------------------------------------
1. record video
2. Edit video
3. Upload video


In [12]:
# 4. Prove persistence across sessions
# unlike state backend, this file survives even after python exits. A brand new agent can read it back from disk

fresh_agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash-lite", backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True)
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim"
    }]
    # no file state is passed in, the file is read straight from disk
})

print("\n ---  Read back with fresh agent to prove disk persistence ---")
print(followup["messages"][-1].content)


 ---  Read back with fresh agent to prove disk persistence ---
The file `/notes/todo.txt` contains the following items:
1. record video
2. Edit video
3. Upload video


#### Deep Agent - StoreBackend Verification
Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do.

In [14]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash-lite",
    backend=StoreBackend(
        # local dev: static namespace, no deployment runtime needed
        # In a langsmith deployment you'd use the user identity version.
        namespace=lambda rt: ("demo-user")
    ),
    store=store
)

print("Agent created with store backend + static namespace")

Agent created with store backend + static namespace


In [15]:
import os
import uuid

# thread 1 - write a file
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you have done it"
            )
        }]
    },
    config=thread_1
)

print("--- Agent reply (thread_1) ---")
print(result["messages"][-1].content)

--- Agent reply (thread_1) ---
I have created the file /notes/todo.txt with the content you provided.


In [16]:
# thread 2 - reda back on different thread
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim"
    }]},
    config=thread_2
)

print(f" ---  read back on a different thread --- ")
print(followup["messages"][-1].content)

 ---  read back on a different thread --- 
The file contains the following todos:
1. Record video
2. Edit video
3. Upload video


### With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

## Deep Agent Backends — Where Your Files Actually Live

Every Deep Agent works with a virtual filesystem: its tools read and write files using paths like /notes/todo.txt. But those paths are an abstraction. The backend decides where that data physically lives — and that single choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

<br>

##### 1. StateBackend (default)
Files live in the agent's LangGraph state — i.e. in RAM, tied to a single thread. They're available during the run via result["files"], and you must manually carry that state forward to keep using them.

- Where: in-memory dict, inside one thread's state
- Survives: within a single conversation/thread only
- Gone when: the thread ends or the process exits
- Use it for: ephemeral scratch space, temporary working files

<br>

##### 2. FilesystemBackend
Files are written to your real disk, under root_dir. With virtual_mode=True, a virtual path like /notes/todo.txt maps to root_dir/notes/todo.txt. These are genuine files you can open in an editor or cat from a terminal.

- Where: your actual filesystem, relative to root_dir
- Survives: across process restarts — it's a real file
- Use it for: when the agent should edit real project files
- ⚠️ Caution: grants the agent real read/write disk access — point it at a scratch directory, not anything important

<br>

##### 3. StoreBackend
Files live in a LangGraph store, scoped by a namespace. This is the only backend whose files persist across threads/conversations — write on one thread, read back on another. With InMemoryStore this works within a single process; swap in a persistent store (e.g. Postgres-backed) for true cross-restart durability.

- Where: inside the store object, under a namespace key
- Survives: across threads sharing the same store
- Gone when: the process exits (if using InMemoryStore) — use a persistent store to outlive restarts
- Use it for: long-term memory, per-user file spaces

<br>

#### Quick comparison
| Backend            | Lives in                | Cross-thread? | Survives restart?       | Real file on disk? |
|--------------------|-------------------------|---------------|--------------------------|--------------------|
| StateBackend       | LangGraph state         | ❌            | ❌                       | ❌                 |
| FilesystemBackend  | Your disk               | ✅            | ✅                       | ✅                 |
| StoreBackend       | A LangGraph store       | ✅            | only w/ persistent store like postgres | ❌                 |

`Mental model: the agent never knows the difference. Its tools always speak in file paths — the backend quietly translates every read/write into the right operation underneath. That's what lets you swap backends without touching a line of agent logic.`